# PlantVillage: all color classes
Train from `raw/color` using the same pipeline as the command-line trainer. Class names come from sorted dataset folders; the output layer matches their count. Images are streamed in batches instead of loading the entire dataset into memory.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from dataset import DEFAULT_DATASET, discover_classes, split_dataset
from train_model import train

dataset_path = DEFAULT_DATASET  # Change this to your PlantVillage raw/color path
classes = discover_classes(dataset_path)
print(f"{len(classes)} classes, {sum(map(len, classes.values())):,} images")


## Class distribution
PlantVillage classes have different image counts. Each class is split into 70% training, 15% validation, and 15% test images using a fixed seed.


In [ ]:
plt.figure(figsize=(12, 10))
plt.barh(list(classes), [len(paths) for paths in classes.values()])
plt.xlabel("Images")
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for axis, (name, paths) in zip(axes.flat, classes.items()):
    axis.imshow(Image.open(paths[0]).convert("RGB"))
    axis.set_title(name.replace("___", ": ").replace("_", " "), fontsize=8)
    axis.axis("off")
plt.tight_layout()


## Train and export
RGB pixels are resized to 128 × 128. A frozen pretrained MobileNetV2 encoder normalizes pixels to [-1, 1] internally and extracts features once. A new classification head learns the dataset classes. Early stopping restores the best validation weights. The held-out test split is evaluated after training.


In [ ]:
model, history, metadata = train(
    dataset_path=dataset_path,
    output_dir=Path.cwd(),
    epochs=30,
    batch_size=32,
    image_size=128,
    seed=42,
)


## Training curves and test accuracy


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric in zip(axes, ["accuracy", "loss"]):
    axis.plot(history.history[metric], label="Training")
    axis.plot(history.history["val_" + metric], label="Validation")
    axis.set_xlabel("Epoch")
    axis.set_ylabel(metric)
    axis.legend()
plt.tight_layout()
print(metadata["test_metrics"])


## Predict a held-out example
Use the class mapping saved with the model. The app uses the same RGB decoding and resize operations.


In [ ]:
import numpy as np
import tensorflow as tf
from train_model import make_dataset

test_paths, test_labels = split_dataset(classes, seed=metadata["seed"])["test"]
images, labels = next(iter(make_dataset(test_paths, test_labels, metadata["image_size"], 16)))
predictions = model.predict(images, verbose=0)
index = 0
plt.imshow(images[index].numpy().astype("uint8"))
plt.axis("off")
print("Actual:", metadata["class_names"][int(labels[index])])
print("Predicted:", metadata["class_names"][int(np.argmax(predictions[index]))])
